# Naive RAG End-to-End Pipeline

A teaching notebook that builds a naive Retrieval-Augmented Generation (RAG)
pipeline step by step, using LangChain wrappers throughout:

1. **Ingestion** — load PDF / URL / text documents, split into chunks.
2. **Embedding + storage** — embed chunks with OpenAI `text-embedding-3-small`,
   store in an in-memory ChromaDB collection.
3. **Query** — embed the user's question with the same embedding model,
   run a similarity search, retrieve the top 3 chunks.
4. **Generation** — pass the retrieved chunks + question to `gpt-4o-mini`,
   strictly instructed to answer using only the retrieved context.

Run the cells top to bottom. Each step is runnable on its own once the
prior cells have executed.

## Setup

Load environment variables and confirm `OPENAI_API_KEY` is present. This
notebook has no mock mode — every embedding and generation call below hits
the real OpenAI API.

In [1]:
import os
from dotenv import load_dotenv

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")  # silence Chroma's telemetry noise
os.environ.setdefault("USER_AGENT", "naive-rag-teaching-demo/1.0")  # silence WebBaseLoader warning
load_dotenv(dotenv_path="../../.env")  # repo-root .env

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to the repo-root .env file "
        "before running this notebook — there is no mock mode."
    )

print("OPENAI_API_KEY loaded:", OPENAI_API_KEY[:7] + "..." )

OPENAI_API_KEY loaded: sk-proj...


## Step (a): Ingestion — load documents and split into chunks

`SOURCES` is a plain list where the user "uploads" documents by pasting
local file paths and/or URLs. This notebook supports three input types
via LangChain's community document loaders:

- `.pdf` → `PyPDFLoader`
- `http(s)://` → `WebBaseLoader`
- anything else (e.g. `.txt`) → `TextLoader`

Each loaded document is then split into overlapping chunks using
`RecursiveCharacterTextSplitter`, which tries to split on paragraph/sentence
boundaries first before falling back to hard character limits — this keeps
chunks semantically coherent for retrieval later.

Two sample local files are included under `data/` so this runs end to end
with no setup. Replace or extend `SOURCES` with your own files/URLs.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# "Upload" step: user sets local file paths and/or URLs here (one or many).
SOURCES = [
    "data/company_handbook.txt",
    "data/product_faq.pdf",
]


def load_source(source: str):
    if source.startswith("http://") or source.startswith("https://"):
        loader = WebBaseLoader(source)
    elif source.lower().endswith(".pdf"):
        loader = PyPDFLoader(source)
    else:
        loader = TextLoader(source, encoding="utf-8")
    return loader.load()


raw_documents = []
for source in SOURCES:
    docs = load_source(source)
    raw_documents.extend(docs)
    print(f"Loaded {len(docs)} document(s) from {source}")

# CHunking 
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_documents)

print(f"\nTotal chunks after splitting: {len(chunks)}")
print("\n--- First chunk preview ---")
print(chunks[0].page_content[:300])

Loaded 1 document(s) from data/company_handbook.txt
Loaded 1 document(s) from data/product_faq.pdf

Total chunks after splitting: 7

--- First chunk preview ---
Acme Robotics Employee Handbook (Sample)

Section 1: Leave Policy
Full-time employees at Acme Robotics accrue 18 days of paid time off (PTO)
per calendar year. PTO accrues at a rate of 1.5 days per completed month
of service. Unused PTO up to 5 days may be carried over into the next
calendar year; a


## Step (b): Embed chunks and store in an in-memory ChromaDB collection

Each chunk is embedded with OpenAI's `text-embedding-3-small` model via
LangChain's `OpenAIEmbeddings` wrapper, then stored in a ChromaDB
collection. This Chroma instance is in-memory only (no `persist_directory`
is set) — it exists for the lifetime of this notebook process and is
rebuilt from scratch every time this cell runs, which keeps the demo
simple and reproducible.

This completes the **ingestion pipeline**.

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)

# In-memory Chroma: no persist_directory, so nothing is written to disk.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="naive_rag_demo",
)

print(f"Stored {vector_store._collection.count()} chunk embeddings in Chroma.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Stored 7 chunk embeddings in Chroma.


## Step (c): Query — embed the question and retrieve the top 3 chunks

The user's question is embedded with the **same** `text-embedding-3-small`
model, then a similarity search runs against the Chroma collection built
above, returning the 3 most similar chunks.

In [ ]:
# User sets their question here.
QUESTION = "How many days of PTO do full-time employees get, and how much can carry over?"

TOP_K = 3
# there is no explicit embedding for question done...because it is something vector databases handles inside it 
retrieved_chunks = vector_store.similarity_search(QUESTION, k=TOP_K)

print(f"Retrieved {len(retrieved_chunks)} chunk(s) for: {QUESTION!r}\n")
for i, chunk in enumerate(retrieved_chunks, start=1):
    source = chunk.metadata.get("source", "unknown")
    print(f"--- Chunk {i} (source: {source}) ---")
    print(chunk.page_content[:300])
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 3 chunk(s) for: 'How many days of PTO do full-time employees get, and how much can carry over?'

--- Chunk 1 (source: data/company_handbook.txt) ---
Acme Robotics Employee Handbook (Sample)

Section 1: Leave Policy
Full-time employees at Acme Robotics accrue 18 days of paid time off (PTO)
per calendar year. PTO accrues at a rate of 1.5 days per completed month
of service. Unused PTO up to 5 days may be carried over into the next
calendar year; a

--- Chunk 2 (source: data/company_handbook.txt) ---
Section 2: Remote Work Policy
Employees may work remotely up to 3 days per week, subject to manager
approval. Fully remote arrangements require VP-level sign-off and are
reviewed every 6 months. All remote employees must be reachable during
core hours, which are 10:00 AM to 4:00 PM in their local ti

--- Chunk 3 (source: data/company_handbook.txt) ---
Section 4: Performance Reviews
Performance reviews are conducted twice a year, in June and December.
Each review includes a self-asse

## Step (d): Generation — answer strictly from retrieved context

The 3 retrieved chunks are stitched into a context block and passed to
`gpt-4o-mini` via LangChain's `ChatOpenAI` wrapper, with a system prompt
that strictly instructs the model to answer **only** using the provided
context, and to say so explicitly if the answer isn't in it (no relying
on outside/parametric knowledge).

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the "
    "context provided below. Do not use any outside knowledge. If the "
    "answer cannot be found in the context, say exactly: "
    "\"I don't have enough information in the provided documents to "
    "answer that.\""
)

context_block = "\n\n".join(
    f"[Chunk {i+1} | source: {c.metadata.get('source', 'unknown')}]\n{c.page_content}"
    for i, c in enumerate(retrieved_chunks)
)

user_message = f"Context:\n{context_block}\n\nQuestion: {QUESTION}"

response = llm.invoke([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=user_message),
])

print("Question:", QUESTION)
print("\nFinal Answer:\n")
print(response.content)

Question: How many days of PTO do full-time employees get, and how much can carry over?

Final Answer:

Full-time employees at Acme Robotics accrue 18 days of paid time off (PTO) per calendar year. They may carry over unused PTO up to 5 days into the next calendar year; any remaining balance above 5 days is forfeited on December 31st.
